# Processador de Dataset de Folhas de Tomate (Versão Google Colab)

Este notebook foi desenvolvido para processar o dataset de folhas de tomate de forma reprodutível, estruturada e perfeitamente balanceada no ambiente do Google Colab. 

### Arquitetura do Pipeline de Dados:
1. **Balanceamento Rigoroso**: Seleção de exatamente 1.000 imagens por classe (10.000 no total) a partir das originais.
2. **Global Test Set**: Separação estratificada de 15% de todas as imagens (150 por classe) para teste final (intocável).
3. **Global Train Pool**: Divisão dos 85% restantes em duas metades exatas: `train_fase_1` (50%, correspondendo a 425 imagens por classe) e `train_fase_2` (50%, correspondendo a 425 imagens por classe).
4. **Subsets Cumulativos (Fase 1)**: Criação de subsets cumulativos com escala logarítmica `[1%, 2%, 5%, 10%, 20%, 50%, 100%]` da `train_fase_1`, seguindo a Regra de Inclusão Cumulativa (dados de 1% dentro de 2%, etc.).
5. **Subset do Futuro**: União de 100% da Fase 1 e 100% da Fase 2 (`fase_1_mais_fase_2`).
6. **dataset_metadata.json**: Geração de arquivo JSON mapeando a linhagem (lineage) de cada arquivo.

## Passo 1: Configuração e Importação de Bibliotecas

In [ ]:
import os
import json
import math
import random
import shutil
from pathlib import Path
from typing import Dict, List, Any

print("✓ Ambiente configurado com bibliotecas padrão.")

## Passo 2: Montar o Google Drive (Opcional)
Se o seu dataset original estiver no Google Drive, descomente e execute a célula abaixo.

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')

## Passo 3: Definir Caminhos de Entrada e Saída
Ajuste os caminhos abaixo conforme a sua estrutura no Colab. Por padrão, leremos de uma pasta local no Colab `/content/dataset_original` e salvaremos em `/content/dataset_processado`.

In [ ]:
# Caminhos recomendados para execução no Google Colab
INPUT_DIR = "/content/dataset_original"
OUTPUT_DIR = "/content/dataset_processado"
SEED = 42
MAX_IMAGES_PER_CLASS = 1000
TEST_PCT = 0.15

print(f"Caminho de Entrada: {INPUT_DIR}")
print(f"Caminho de Saída: {OUTPUT_DIR}")
print(f"Imagens por Classe (Balanceado): {MAX_IMAGES_PER_CLASS}")

## Passo 4: Código Principal de Processamento

In [ ]:
import json
import shutil
def process_dataset(input_dir_str: str, output_dir_str: str, seed: int, max_images_per_class: int, test_pct: float):
    input_path = Path(input_dir_str).resolve()
    output_path = Path(output_dir_str).resolve()

    if not input_path.exists():
        print(f"[!] Caminho de entrada não encontrado: {input_path}")
        print("Crie a pasta ou faça upload do dataset antes de prosseguir.")
        return False

    class_dirs = [d for d in input_path.iterdir() if d.is_dir()]
    if not class_dirs:
        raise ValueError(f"Nenhuma pasta de classe encontrada em {input_path}")

    extensions = ("*.jpg", "*.jpeg", "*.png", "*.JPG", "*.PNG", "*.JPEG")

    metadata = {
        "config": {
            "seed": seed,
            "max_images_per_class": max_images_per_class,
            "test_pct": test_pct,
            "input_dir": str(input_path),
            "output_dir": str(output_path),
        },
        "statistics": {
            "total_images_processed": 0,
            "classes": {},
            "splits": {
                "global_test": 0,
                "train_fase_1": 0,
                "train_fase_2": 0,
                "fase_1_mais_fase_2": 0,
                "subsets_fase_1": {}
            }
        },
        "file_lineage": {}
    }

    cumulative_percentages = [1, 2, 5, 10, 20, 50, 100]
    for pct in cumulative_percentages:
        metadata["statistics"]["splits"]["subsets_fase_1"][f"pct_{pct}"] = 0

    global_test_dir = output_path / "global_test"
    train_fase_1_dir = output_path / "train_fase_1"
    train_fase_2_dir = output_path / "train_fase_2"
    fase_1_plus_2_dir = output_path / "fase_1_mais_fase_2"
    subsets_dir = output_path / "subsets_fase_1"

    # Explicitly create all base split directories
    global_test_dir.mkdir(parents=True, exist_ok=True)
    train_fase_1_dir.mkdir(parents=True, exist_ok=True)
    train_fase_2_dir.mkdir(parents=True, exist_ok=True)
    fase_1_plus_2_dir.mkdir(parents=True, exist_ok=True)
    subsets_dir.mkdir(parents=True, exist_ok=True)

    # Ensure all percentage subdirectories within subsets_fase_1 exist
    for pct in cumulative_percentages:
        (subsets_dir / f"pct_{pct}").mkdir(parents=True, exist_ok=True)

    rng = random.Random(seed)

    for class_dir in class_dirs:
        class_name = class_dir.name
        images = []
        for ext in extensions:
            images.extend(list(class_dir.glob(ext)))

        images = sorted(list(set(images)))
        total_found = len(images)

        if total_found == 0:
            continue

        shuffled_original = list(images)
        rng.shuffle(shuffled_original)
        selected_images = shuffled_original[:max_images_per_class]
        actual_class_count = len(selected_images)

        metadata["statistics"]["classes"][class_name] = {
            "available_count": total_found,
            "selected_count": actual_class_count,
            "splits": {
                "global_test": 0,
                "train_fase_1": 0,
                "train_fase_2": 0,
                "fase_1_mais_fase_2": 0,
                "subsets_fase_1": {}
            }
        }
        for pct in cumulative_percentages:
            metadata["statistics"]["classes"][class_name]["splits"]["subsets_fase_1"][f"pct_{pct}"] = 0

        test_size = int(math.ceil(actual_class_count * test_pct))
        test_images = selected_images[:test_size]
        train_pool_images = selected_images[test_size:]

        half_pool = len(train_pool_images) // 2
        fase_1_images = train_pool_images[:half_pool]
        fase_2_images = train_pool_images[half_pool:]

        metadata["statistics"]["classes"][class_name]["splits"]["global_test"] = len(test_images)
        metadata["statistics"]["classes"][class_name]["splits"]["train_fase_1"] = len(fase_1_images)
        metadata["statistics"]["classes"][class_name]["splits"]["train_fase_2"] = len(fase_2_images)
        metadata["statistics"]["classes"][class_name]["splits"]["fase_1_mais_fase_2"] = len(fase_1_images) + len(fase_2_images)

        metadata["statistics"]["splits"]["global_test"] += len(test_images)
        metadata["statistics"]["splits"]["train_fase_1"] += len(fase_1_images)
        metadata["statistics"]["splits"]["train_fase_2"] += len(fase_2_images)
        metadata["statistics"]["splits"]["fase_1_mais_fase_2"] += len(fase_1_images) + len(fase_2_images)
        metadata["statistics"]["total_images_processed"] += actual_class_count

        def copy_image(src_path, dest_dir, rel_copied_paths):
            dest_file_path = dest_dir / class_name / src_path.name
            rel_dest_path = dest_file_path.relative_to(output_path).as_posix()
            rel_copied_paths.append(rel_dest_path)

            (dest_dir / class_name).mkdir(parents=True, exist_ok=True)
            shutil.copy2(src_path, dest_file_path)

        for img in selected_images:
            metadata["file_lineage"][img.name] = {
                "original_path": img.relative_to(input_path.parent).as_posix(),
                "class": class_name,
                "destinations": []
            }

        for img in test_images:
            copy_image(img, global_test_dir, metadata["file_lineage"][img.name]["destinations"])
            metadata["file_lineage"][img.name]["split"] = "global_test"

        for img in fase_1_images:
            copy_image(img, train_fase_1_dir, metadata["file_lineage"][img.name]["destinations"])
            metadata["file_lineage"][img.name]["split"] = "train_fase_1"
            copy_image(img, fase_1_plus_2_dir, metadata["file_lineage"][img.name]["destinations"])

        for img in fase_2_images:
            copy_image(img, train_fase_2_dir, metadata["file_lineage"][img.name]["destinations"])
            metadata["file_lineage"][img.name]["split"] = "train_fase_2"
            copy_image(img, fase_1_plus_2_dir, metadata["file_lineage"][img.name]["destinations"])

        for pct in cumulative_percentages:
            subset_size = int(round(len(fase_1_images) * (pct / 100.0)))
            subset_images = fase_1_images[:subset_size]

            metadata["statistics"]["classes"][class_name]["splits"]["subsets_fase_1"][f"pct_{pct}"] = len(subset_images)
            metadata["statistics"]["splits"]["subsets_fase_1"][f"pct_{pct}"] += len(subset_images)

            subset_pct_dir = subsets_dir / f"pct_{pct}"
            # The directory subset_pct_dir is now guaranteed to exist from the earlier block
            for img in subset_images:
                copy_image(img, subset_pct_dir, metadata["file_lineage"][img.name]["destinations"])

    metadata_file_path = output_path / "dataset_metadata.json"
    output_path.mkdir(parents=True, exist_ok=True)
    with open(metadata_file_path, "w", encoding="utf-8") as f:
        json.dump(metadata, f, indent=4, ensure_ascii=False)

    print(f"\n[+] Separação Concluída. Estatísticas gravadas em: {metadata_file_path}")
    return True

print("A função 'process_dataset' foi definida. Ela será executada posteriormente no notebook, na célula 'MVq9awxZIomm'.")

## Passo 5: Executar o Processamento

In [ ]:
# Execute esta célula para processar o dataset
process_dataset(
    input_dir_str=INPUT_DIR,
    output_dir_str=OUTPUT_DIR,
    seed=SEED,
    max_images_per_class=MAX_IMAGES_PER_CLASS,
    test_pct=TEST_PCT
)

## Passo 6: Compactar os Resultados para Download
Gere um arquivo `.zip` com todo o dataset processado para que você possa baixá-lo rapidamente.

In [ ]:
!zip -q -r tomato_processed_dataset.zip /content/dataset_processado
print("✓ Dataset processado e compactado em 'tomato_processed_dataset.zip'")